# 🌸 专属 AI 娘云端全能算力工坊 (Google Colab 一键部署 · v5 旗舰版)

专为 **95GB RTX PRO 6000 / A100 / L4 / 免费 T4** 打造的【云烘本地用】全能云端算力服务端：
1. **真 GPU 零样本声音克隆 (`F5-TTS`) + 8 位幼态慢速 `Edge-TTS` 无缝回退**：锁定参考干声后直接真 GPU 零样本克隆并输出 48kHz 高清 MP3；支持本地一键批量烘焙 48 句离线语音库，关掉 Colab 也能 0 延迟秒放！
2. **1080×1920 治愈系田园庭院一键云烘 (`POST /v1/scene/bake`)**：95GB 大显存自动启用 `cagliostrolab/animagine-xl-3.1`（小显存或异常自动回退 `stabilityai/sdxl-turbo`），10 秒生成宫崎骏/蔚蓝档案风白天暖阳 + 狼人杀星光夜景，下载到本地永久复用！
3. **BiRefNet GPU 发丝级透明抠图 + Rembg 自动回退 (`POST /v1/matting/birefnet`)**：全异步线程不阻塞心跳，永不因缺库或显存抖动崩掉。
4. **Cloudflare 免费公网隧道 + 本地回环健康心跳自检 + 断线自动自愈**：复制隧道地址与 Token 粘贴回本地【Colab 算力控制台】即可一键起飞！

In [ ]:
# @title 1️⃣ 检查 GPU 算力并安装 v5 全能依赖（F5-TTS + Diffusers 场景烘焙 + BiRefNet + Rembg）
!nvidia-smi
!which ffmpeg >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y ffmpeg)
!pip install -q fastapi uvicorn python-multipart pillow edge-tts httpx soundfile diffusers accelerate safetensors transformers torch torchvision timm einops kornia scikit-image rembg
!pip install -q f5-tts || echo '⚠️ f5-tts 可选包安装遇到环境差异，已保留幼态慢速 Edge-TTS 无缝兜底'
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print('✅ v5 旗舰全能环境（F5-TTS / Diffusers 场景云烘 / BiRefNet / Rembg / ffmpeg / cloudflared）安装完成！')

In [ ]:
# @title 2️⃣ 启动 v5 旗舰全能服务端【F5-TTS真克隆 + 1080x1920场景云烘 + BiRefNet抠图 + 心跳自检】
import asyncio
import io
import os
import re
import secrets
import shutil
import subprocess
import threading
import time
import traceback
import uuid
from datetime import datetime
from pathlib import Path
from typing import Optional, Tuple

import edge_tts
import httpx
import torch
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
from PIL import Image, ImageFilter
from pydantic import BaseModel

API_TOKEN = secrets.token_urlsafe(18)
REF_DIR = Path('/content/voice_refs')
REF_DIR.mkdir(parents=True, exist_ok=True)
SCENE_CACHE_DIR = Path('/content/baked_scenes')
SCENE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
BOOT_TIME = time.time()
HAS_FFMPEG = shutil.which('ffmpeg') is not None

app = FastAPI(title='Personal AI Girls Colab Flagship Studio v5')

# GPU 线程安全互斥锁（配合 asyncio.to_thread，重型推理绝不阻塞 /health 心跳！）
gpu_tts_lock = threading.Lock()
gpu_scene_lock = threading.Lock()
gpu_matting_lock = threading.Lock()

# 三大核心引擎运行态
f5tts_model = None
f5tts_loaded = False
f5tts_failed_permanently = False
f5tts_status = 'standby_on_demand'
f5tts_last_error = ''

scene_pipe = None
scene_pipe_kind = ''
scene_status = 'standby_on_demand'
scene_last_error = ''

birefnet_model = None
birefnet_loaded = False
birefnet_status = 'standby_on_demand'
birefnet_last_error = ''

stats = {
    'tts_requests': 0,
    'f5tts_clones': 0,
    'edge_fallbacks': 0,
    'scene_bakes': 0,
    'matting_requests': 0,
    'health_checks': 0,
}


def now_str() -> str:
    return datetime.now().strftime('%H:%M:%S')


def get_gpu_stats() -> dict:
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        used = torch.cuda.memory_allocated(0) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(0) / (1024 ** 3)
        total = props.total_memory / (1024 ** 3)
        return {
            'gpu_name': props.name,
            'vram_used_gb': round(max(used, reserved), 2),
            'vram_total_gb': round(total, 2),
        }
    return {'gpu_name': 'CPU Mode (建议在 Colab 顶部选择 GPU 运行时)', 'vram_used_gb': 0.0, 'vram_total_gb': 0.0}


def verify_auth(authorization: str = '') -> None:
    if authorization and authorization.replace('Bearer ', '').strip() == API_TOKEN:
        return
    raise HTTPException(status_code=401, detail='Invalid API Token')


def list_locked_voices() -> list:
    found = set()
    for ext in ('*.wav', '*.mp3', '*.flac', '*.ogg', '*.m4a'):
        for p in REF_DIR.glob(ext):
            stem = p.name[:-9] if p.name.endswith('.norm.wav') else p.stem
            if stem:
                found.add(stem)
    return sorted(found)


def find_character_ref(character_id: str) -> Optional[Path]:
    safe_id = Path(character_id).name
    norm_wav = REF_DIR / f'{safe_id}.norm.wav'
    if norm_wav.exists() and norm_wav.stat().st_size > 1000:
        return norm_wav
    for ext in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'):
        cand = REF_DIR / f'{safe_id}{ext}'
        if cand.exists() and cand.stat().st_size > 1000:
            return cand
    return None


@app.get('/health')
async def health():
    stats['health_checks'] += 1
    return {
        'ok': True,
        'version': 'v5',
        'uptime_sec': int(time.time() - BOOT_TIME),
        'ffmpeg_ready': HAS_FFMPEG,
        'gpu': get_gpu_stats(),
        'models': {
            'voice_engine': f'F5-TTS GPU ({f5tts_status}) + Edge-TTS Fallback',
            'f5tts_status': f5tts_status,
            'f5tts_last_error': f5tts_last_error,
            'scene_baker': f'{scene_pipe_kind or "animagine-xl / sdxl-turbo"} ({scene_status})',
            'scene_status': scene_status,
            'scene_last_error': scene_last_error,
            'birefnet_matting': birefnet_status,
            'birefnet_last_error': birefnet_last_error,
        },
        'stats': dict(stats),
        'locked_voices': list_locked_voices(),
    }


# ==================== 1. 参考干声锁定 + 真 GPU 零样本声音克隆 (F5-TTS) ====================
@app.post('/v1/voices/lock/{character_id}')
async def lock_reference_voice(
    character_id: str,
    file: UploadFile = File(...),
    ref_text: str = Form(default=''),
    authorization: str = Header(default=''),
):
    verify_auth(authorization)
    safe_id = Path(character_id).name
    ext = (Path(file.filename or 'ref.wav').suffix or '.wav').lower()
    if ext not in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'):
        ext = '.wav'

    # 清理该角色旧格式干声，避免后缀冲突
    for old_ext in ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.norm.wav'):
        (REF_DIR / f'{safe_id}{old_ext}').unlink(missing_ok=True)

    dst = REF_DIR / f'{safe_id}{ext}'
    dst.write_bytes(await file.read())

    txt_path = REF_DIR / f'{safe_id}.txt'
    if ref_text.strip():
        txt_path.write_text(ref_text.strip(), encoding='utf-8')
    else:
        txt_path.unlink(missing_ok=True)

    # 预转码为标准 24kHz 单声道 WAV，并裁至最多 12 秒黄金参考片段（F5-TTS 效果最稳）
    normalized = False
    if HAS_FFMPEG:
        norm_wav = REF_DIR / f'{safe_id}.norm.wav'
        cmd = [
            'ffmpeg', '-y', '-i', str(dst),
            '-t', '12', '-ac', '1', '-ar', '24000',
            str(norm_wav),
        ]
        proc = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        normalized = proc.returncode == 0 and norm_wav.exists() and norm_wav.stat().st_size > 1000

    print(f'[{now_str()}] 🎙️ 已锁定参考干声: {safe_id} -> {dst.name} (24kHz标准化={normalized})')
    return {
        'ok': True,
        'character_id': safe_id,
        'saved': dst.name,
        'normalized_wav': normalized,
        'locked_voices': list_locked_voices(),
    }


class SpeechReq(BaseModel):
    model: str = 'f5-tts'
    input: str = ''
    text: str = ''
    voice: str = 'deepseek'
    emotion: str = 'happy'
    voice_prompt: str = ''
    response_format: str = 'mp3'


# 8 位全明星 AI 娘幼萌慢速神经网络兜底音色映射 (voice, base_rate, base_pitch)
VOICE_FALLBACK = {
    'gpt':      ('zh-CN-XiaoxiaoNeural',        -11, 11),
    'claude':   ('zh-TW-HsiaoYuNeural',         -13, 11),
    'deepseek': ('zh-CN-YunxiaNeural',          -10, 13),
    'gemini':   ('zh-TW-HsiaoChenNeural',       -10, 14),
    'zhipu':    ('zh-CN-XiaoyiNeural',          -14,  9),
    'grok':     ('zh-CN-liaoning-XiaobeiNeural', -9, 14),
    'kimi':     ('zh-CN-XiaoxiaoNeural',        -12, 15),
    'qwen':     ('zh-CN-shaanxi-XiaoniNeural',  -11, 12),
}

EMOTION_DELTA = {
    'idle':     (0, 0, 0.94),
    'talking':  (0, 2, 0.94),
    'happy':    (3, 4, 0.98),
    'smug':     (-2, 3, 0.93),
    'angry':    (4, 5, 1.00),
    'sad':      (-5, -2, 0.88),
    'shy':      (-5, 4, 0.89),
    'shocked':  (4, 6, 0.99),
    'thinking': (-6, 1, 0.88),
    'dizzy':    (-6, -3, 0.88),
}


def _load_f5tts_safe() -> bool:
    global f5tts_model, f5tts_loaded, f5tts_failed_permanently, f5tts_status, f5tts_last_error
    if f5tts_loaded and f5tts_model is not None:
        return True
    if f5tts_failed_permanently:
        return False
    if not torch.cuda.is_available():
        f5tts_status = 'cpu_fallback_edge'
        return False
    try:
        f5tts_status = 'loading'
        print(f'[{now_str()}] 🔄 首次按需加载 F5-TTS GPU 零样本声音克隆引擎...')
        from f5_tts.api import F5TTS
        f5tts_model = F5TTS(device='cuda')
        f5tts_loaded = True
        f5tts_status = 'loaded'
        f5tts_last_error = ''
        print(f'[{now_str()}] ✅ F5-TTS GPU 零样本声音克隆模型加载完成！')
        return True
    except Exception as exc:
        f5tts_loaded = False
        f5tts_status = 'fallback_edge'
        f5tts_last_error = f'{type(exc).__name__}: {str(exc)[:180]}'
        if isinstance(exc, ImportError):
            f5tts_failed_permanently = True
        print(f'[{now_str()}] ⚠️ F5-TTS 加载失败，无缝回退幼态 Edge-TTS: {f5tts_last_error}')
        return False


def _synthesize_f5tts_sync(text: str, character_id: str, emotion: str, want_mp3: bool) -> Optional[Tuple[bytes, str]]:
    global f5tts_last_error
    ref_path = find_character_ref(character_id)
    if ref_path is None:
        return None
    with gpu_tts_lock:
        if not _load_f5tts_safe():
            return None
        txt_path = REF_DIR / f'{Path(character_id).name}.txt'
        ref_text = txt_path.read_text(encoding='utf-8').strip() if txt_path.exists() else ''
        _, _, speed = EMOTION_DELTA.get(emotion, EMOTION_DELTA['talking'])
        uid = uuid.uuid4().hex[:8]
        tmp_wav = Path(f'/tmp/f5_{uid}.wav')
        tmp_mp3 = Path(f'/tmp/f5_{uid}.mp3')
        try:
            try:
                f5tts_model.infer(
                    ref_file=str(ref_path),
                    ref_text=ref_text,
                    gen_text=text,
                    file_wave=str(tmp_wav),
                    speed=speed,
                    remove_silence=True,
                )
            except TypeError:
                f5tts_model.infer(
                    ref_file=str(ref_path),
                    ref_text=ref_text,
                    gen_text=text,
                    file_wave=str(tmp_wav),
                )
            if not tmp_wav.exists() or tmp_wav.stat().st_size < 1000:
                return None
            if want_mp3 and HAS_FFMPEG:
                proc = subprocess.run(
                    ['ffmpeg', '-y', '-i', str(tmp_wav), '-ar', '48000', '-ac', '1', '-b:a', '192k', str(tmp_mp3)],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.DEVNULL,
                )
                if proc.returncode == 0 and tmp_mp3.exists() and tmp_mp3.stat().st_size > 1000:
                    return tmp_mp3.read_bytes(), 'audio/mpeg'
            return tmp_wav.read_bytes(), 'audio/wav'
        except Exception as exc:
            f5tts_last_error = f'{type(exc).__name__}: {str(exc)[:180]}'
            print(f'[{now_str()}] ⚠️ F5-TTS 推理异常，回退幼态 Edge-TTS: {f5tts_last_error}')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return None
        finally:
            tmp_wav.unlink(missing_ok=True)
            tmp_mp3.unlink(missing_ok=True)


@app.post('/v1/audio/speech')
async def synthesize(req: SpeechReq, authorization: str = Header(default='')):
    verify_auth(authorization)
    content = (req.input or req.text).strip()
    if not content:
        raise HTTPException(status_code=400, detail='Empty text')
    stats['tts_requests'] += 1

    # 1. 若该角色已锁定参考干声，优先使用 F5-TTS GPU 真零样本声音克隆
    want_mp3 = (req.response_format or 'mp3').lower() != 'wav'
    cloned = await asyncio.to_thread(_synthesize_f5tts_sync, content, req.voice, req.emotion, want_mp3)
    if cloned is not None:
        data, mime = cloned
        stats['f5tts_clones'] += 1
        return Response(content=data, media_type=mime, headers={'X-TTS-Engine': 'f5-tts-gpu'})

    # 2. 无参考干声或加载中：无缝回退幼态慢速 Edge-TTS（绝不报错！）
    stats['edge_fallbacks'] += 1
    voice_name, base_r, base_p = VOICE_FALLBACK.get(req.voice, VOICE_FALLBACK['deepseek'])
    dr, dp, _ = EMOTION_DELTA.get(req.emotion, EMOTION_DELTA['talking'])
    final_r = max(-22, min(4, base_r + dr))
    final_p = max(3, min(20, base_p + dp))
    rate_str = f'+{final_r}%' if final_r >= 0 else f'{final_r}%'
    pitch_str = f'+{final_p}Hz' if final_p >= 0 else f'{final_p}Hz'

    tmp = Path(f'/tmp/tts_{uuid.uuid4().hex[:8]}.mp3')
    try:
        comm = edge_tts.Communicate(content, voice=voice_name, rate=rate_str, pitch=pitch_str)
        await comm.save(str(tmp))
        data = tmp.read_bytes()
        return Response(content=data, media_type='audio/mpeg', headers={'X-TTS-Engine': 'edge-tts-fallback'})
    except Exception as exc:
        print(f'[{now_str()}] ⚠️ Edge-TTS 兜底合成失败: {exc}')
        raise HTTPException(status_code=500, detail=f'TTS failed: {exc}')
    finally:
        tmp.unlink(missing_ok=True)


# ==================== 2. 95GB GPU 1080x1920 治愈田园庭院场景一键云烘 ====================
class SceneBakeReq(BaseModel):
    time_of_day: str = 'day'  # 'day' | 'night'
    width: int = 1080
    height: int = 1920
    style: str = 'ghibli_blue_archive'
    prompt_extra: str = ''
    seed: int = -1


def _load_scene_pipeline_safe() -> bool:
    global scene_pipe, scene_pipe_kind, scene_status, scene_last_error
    if scene_pipe is not None:
        return True
    if not torch.cuda.is_available():
        scene_status = 'no_gpu'
        scene_last_error = 'CUDA GPU 不可用，请在 Colab 顶部切换 GPU 运行时'
        return False

    vram_total = get_gpu_stats().get('vram_total_gb', 0.0)
    from diffusers import AutoPipelineForText2Image, EulerAncestralDiscreteScheduler

    # 95GB RTX PRO 6000 / A100 / L4 (>=20GB) 优先启用 Animagine XL 3.1 二次元治愈大作；否则自动用 SDXL-Turbo
    candidates = (
        [('animagine-xl-3.1', 'cagliostrolab/animagine-xl-3.1'), ('sdxl-turbo', 'stabilityai/sdxl-turbo')]
        if vram_total >= 20.0
        else [('sdxl-turbo', 'stabilityai/sdxl-turbo')]
    )
    for kind, repo_id in candidates:
        try:
            scene_status = f'loading_{kind}'
            print(f'[{now_str()}] 🎨 正在加载场景烘焙模型 {repo_id} (显存 {vram_total:.0f}GB)...')
            kwargs = {'torch_dtype': torch.float16, 'use_safetensors': True}
            if kind == 'sdxl-turbo':
                kwargs['variant'] = 'fp16'
            pipe = AutoPipelineForText2Image.from_pretrained(repo_id, **kwargs)
            if kind == 'animagine-xl-3.1':
                pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
            pipe = pipe.to('cuda')
            try:
                pipe.enable_vae_slicing()
            except Exception:
                pass
            scene_pipe = pipe
            scene_pipe_kind = kind
            scene_status = 'loaded'
            scene_last_error = ''
            print(f'[{now_str()}] ✅ 场景云烘模型 {kind} 加载完成！')
            return True
        except Exception as exc:
            scene_last_error = f'{kind} load failed: {type(exc).__name__}: {str(exc)[:180]}'
            print(f'[{now_str()}] ⚠️ {scene_last_error}')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    scene_status = 'error'
    return False


def _bake_scene_sync(req: SceneBakeReq) -> Tuple[bytes, str]:
    tod = 'night' if (req.time_of_day or '').strip().lower() == 'night' else 'day'
    target_w = max(512, min(1440, int(req.width or 1080)))
    target_h = max(768, min(2560, int(req.height or 1920)))

    if tod == 'night':
        base_prompt = (
            'masterpiece, best quality, anime background illustration, studio ghibli style, blue archive background art, '
            'cozy sunlit fairytale cottage garden at starry night, glowing golden fireflies floating in air, '
            'deep twilight blue sky with bright stars and crescent moon, warm glowing cottage windows and windmill in upper background, '
            'blooming sakura cherry blossom tree on upper left, colorful flower bed on middle right, '
            'wide open peaceful moonlit green grass lawn courtyard in lower foreground with low wooden fence, '
            'soft healing magical lighting, ultra detailed painterly scenery, no humans, nobody, empty garden'
        )
    else:
        base_prompt = (
            'masterpiece, best quality, anime background illustration, studio ghibli style, blue archive background art, '
            'bright warm morning sunlight, cozy fairytale countryside garden courtyard, '
            'vivid blue sky with fluffy white cumulus clouds, cute wooden cottage and gentle windmill in upper distance, '
            'blooming pink sakura cherry blossom tree on upper left with drifting petals, lush flowerbed on middle right, '
            'wide open flat vibrant sunlit green grass lawn in lower half foreground with low white wooden garden fence, '
            'crisp healing pastoral atmosphere, Makoto Shinkai sky, ultra detailed scenery, no humans, nobody, empty courtyard'
        )
    if req.prompt_extra.strip():
        base_prompt = f'{base_prompt}, {req.prompt_extra.strip()}'

    neg_prompt = (
        '1girl, 2girls, multiple girls, 1boy, people, human, person, character, anime girl, chibi, crowd, '
        'text, watermark, signature, logo, ui, frame, border, lowres, blurry, bad anatomy, dark muddy, photorealistic'
    )

    with gpu_scene_lock:
        if not _load_scene_pipeline_safe():
            raise RuntimeError(scene_last_error or 'Scene pipeline unavailable')

        seed = req.seed if req.seed >= 0 else secrets.randbelow(2 ** 31 - 1)
        gen = torch.Generator(device='cuda').manual_seed(seed)
        t0 = time.time()
        with torch.inference_mode():
            if scene_pipe_kind == 'animagine-xl-3.1':
                img = scene_pipe(
                    prompt=base_prompt,
                    negative_prompt=neg_prompt,
                    width=768,
                    height=1344,
                    num_inference_steps=22,
                    guidance_scale=6.5,
                    generator=gen,
                ).images[0]
            else:
                img = scene_pipe(
                    prompt=base_prompt,
                    width=768,
                    height=1344,
                    num_inference_steps=4,
                    guidance_scale=0.0,
                    generator=gen,
                ).images[0]

        img = img.convert('RGB').resize((target_w, target_h), Image.LANCZOS)
        cache_path = SCENE_CACHE_DIR / f'courtyard_{tod}.jpg'
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=93, subsampling=0)
        jpeg_bytes = buf.getvalue()
        cache_path.write_bytes(jpeg_bytes)
        elapsed = round(time.time() - t0, 2)
        print(f'[{now_str()}] 🖼️ 场景云烘完成 ({scene_pipe_kind}, {tod}, {target_w}x{target_h}, 耗时 {elapsed}s, seed={seed})')
        return jpeg_bytes, scene_pipe_kind


@app.post('/v1/scene/bake')
async def bake_scene(req: SceneBakeReq, authorization: str = Header(default='')):
    verify_auth(authorization)
    stats['scene_bakes'] += 1
    try:
        jpeg_bytes, engine_used = await asyncio.to_thread(_bake_scene_sync, req)
        return Response(
            content=jpeg_bytes,
            media_type='image/jpeg',
            headers={
                'X-Scene-Engine': engine_used,
                'X-Time-Of-Day': req.time_of_day,
            },
        )
    except Exception as exc:
        traceback.print_exc()
        raise HTTPException(status_code=503, detail=f'Scene bake failed: {exc}')


# ==================== 3. BiRefNet GPU 发丝级抠图 + Rembg 自动回退 ====================
def _load_birefnet_safe() -> bool:
    global birefnet_model, birefnet_loaded, birefnet_status, birefnet_last_error
    if birefnet_loaded and birefnet_model is not None:
        return True
    if not torch.cuda.is_available():
        birefnet_status = 'cpu_fallback_rembg'
        return False
    try:
        birefnet_status = 'loading'
        print(f'[{now_str()}] 🔄 正在加载 ZhengPeng7/BiRefNet GPU 抠图模型...')
        from transformers import AutoModelForImageSegmentation
        birefnet_model = AutoModelForImageSegmentation.from_pretrained(
            'ZhengPeng7/BiRefNet', trust_remote_code=True
        ).cuda().eval()
        birefnet_loaded = True
        birefnet_status = 'loaded'
        birefnet_last_error = ''
        print(f'[{now_str()}] ✅ BiRefNet GPU 模型加载完成！')
        return True
    except Exception as exc:
        birefnet_loaded = False
        birefnet_status = 'fallback_rembg'
        birefnet_last_error = f'{type(exc).__name__}: {str(exc)[:180]}'
        print(f'[{now_str()}] ⚠️ BiRefNet 加载失败，自动回退 rembg: {birefnet_last_error}')
        return False


def _matting_sync(raw_bytes: bytes) -> Tuple[bytes, str, tuple]:
    global birefnet_status, birefnet_last_error
    im = Image.open(io.BytesIO(raw_bytes)).convert('RGB')
    engine_used = 'raw-fallback'
    out = None

    with gpu_matting_lock:
        if _load_birefnet_safe():
            try:
                from torchvision import transforms
                tf = transforms.Compose([
                    transforms.Resize((1024, 1024)),
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
                ])
                inp = tf(im).unsqueeze(0).cuda()
                with torch.inference_mode():
                    pred = birefnet_model(inp)[-1].sigmoid().cpu()[0].squeeze()
                mask = transforms.ToPILImage()(pred).resize(im.size, Image.LANCZOS)
                mask = mask.filter(ImageFilter.SMOOTH)
                out = im.convert('RGBA')
                out.putalpha(mask)
                engine_used = 'birefnet-gpu'
            except Exception as exc:
                birefnet_last_error = f'{type(exc).__name__}: {str(exc)[:180]}'
                birefnet_status = 'fallback_rembg'
                print(f'[{now_str()}] ⚠️ BiRefNet 推理异常，切换 rembg 兜底: {birefnet_last_error}')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    if out is None:
        try:
            from rembg import remove as rembg_remove
            res = rembg_remove(im)
            out = res.convert('RGBA') if isinstance(res, Image.Image) else Image.open(io.BytesIO(res)).convert('RGBA')
            engine_used = 'rembg-fallback'
        except Exception as exc:
            print(f'[{now_str()}] ⚠️ rembg 兜底亦异常: {exc}')
            out = im.convert('RGBA')
            engine_used = 'raw-fallback'

    buf = io.BytesIO()
    out.save(buf, format='PNG')
    return buf.getvalue(), engine_used, im.size


@app.post('/v1/matting/birefnet')
async def birefnet_cutout(file: UploadFile = File(...), authorization: str = Header(default='')):
    verify_auth(authorization)
    stats['matting_requests'] += 1
    raw_bytes = await file.read()
    try:
        png_bytes, engine_used, size = await asyncio.to_thread(_matting_sync, raw_bytes)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=f'Invalid image or matting error: {exc}')
    print(f'[{now_str()}] ✂️ 抠图完成 ({engine_used}), 尺寸={size}')
    return Response(
        content=png_bytes,
        media_type='image/png',
        headers={'X-Matting-Engine': engine_used},
    )


# ==================== 4. Cloudflare 免费公网隧道 + 本地回环健康心跳自检 ====================
def start_tunnel() -> Tuple[subprocess.Popen, str]:
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url = ''
    for _ in range(90):
        line = proc.stdout.readline()
        if not line:
            break
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    return proc, url


threading.Thread(
    target=lambda: uvicorn.run(app, host='127.0.0.1', port=7860, log_level='warning'),
    daemon=True,
).start()
time.sleep(2.5)
tunnel_proc, public_url = start_tunnel()
gpu_now = get_gpu_stats()

print('=' * 76)
print('🎉 你的专属 Colab 全能算力工坊 (v5 旗舰版) 已上线！')
print(f"🖥️ 当前 GPU: {gpu_now['gpu_name']} (显存: {gpu_now['vram_used_gb']} / {gpu_now['vram_total_gb']} GB)")
print(f'🧩 能力清单: F5-TTS真克隆 | 1080x1920场景云烘(/v1/scene/bake) | BiRefNet+Rembg抠图 | ffmpeg={HAS_FFMPEG}')
print('🔗 请复制以下地址粘贴到本地【Colab 算力控制台】:')
print('   Colab 服务地址:', public_url or '（隧道获取超时，请重新运行本格）')
print('   专属安全 Token:', API_TOKEN)
print('=' * 76)

heartbeat_idx = 0
while True:
    time.sleep(60)
    heartbeat_idx += 1
    if tunnel_proc.poll() is not None:
        print(f'[{now_str()}] 🔄 检测到 Cloudflare 隧道中断，正在自动重连...')
        tunnel_proc, public_url = start_tunnel()
        print(f'[{now_str()}] ✅ 新隧道地址已上线: {public_url}')

    local_ok = False
    try:
        r = httpx.get('http://127.0.0.1:7860/health', timeout=4.0)
        local_ok = r.status_code == 200
    except Exception:
        local_ok = False

    g = get_gpu_stats()
    uptime_min = int((time.time() - BOOT_TIME) // 60)
    print(
        f"[{now_str()}] 💓 心跳 #{heartbeat_idx} | 自检={'OK' if local_ok else 'WARN'} | 运行 {uptime_min}m | "
        f"GPU: {g['gpu_name']} ({g['vram_used_gb']}/{g['vram_total_gb']}GB) | "
        f"F5={f5tts_status} 场景={scene_status} 抠图={birefnet_status} | "
        f"TTS={stats['tts_requests']}(克隆{stats['f5tts_clones']}) 场景={stats['scene_bakes']} 抠图={stats['matting_requests']} | "
        f"隧道: {public_url}"
    )
